In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.pipeline import DATA

src_csv = r"C:\Users\HP\Desktop\Ejust\Projects\Data Security\nids\data\fe6cb615d161452c_MOHANAD_A4706\data\NF-UNSW-NB15-v2.csv"

expected = [c for c in pq.ParquetFile(DATA / "train.parquet").schema.names if c != "row_id"]
header = pd.read_csv(src_csv, nrows=0).columns.tolist()
print("missing columns:", sorted(set(expected) - set(header)))
print("extra columns:  ", sorted(set(header) - set(expected)))

missing columns: []
extra columns:   []


In [2]:
str_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Attack"]
dtypes = {c: "float64" for c in expected if c not in str_cols}
dtypes.update({c: "string" for c in str_cols})

writer, row0 = None, 0
for chunk in pd.read_csv(src_csv, chunksize=500_000, dtype=dtypes, usecols=expected):
    chunk = chunk[expected]
    chunk.insert(0, "row_id", np.arange(row0, row0 + len(chunk)))
    table = pa.Table.from_pandas(chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(DATA / "unsw.parquet", table.schema, compression="zstd")
    writer.write_table(table)
    row0 += len(chunk)
writer.close()

u = pd.read_parquet(DATA / "unsw.parquet", columns=["IPV4_SRC_ADDR", "Attack"])
print("rows:", len(u), "| source IPs:", u["IPV4_SRC_ADDR"].nunique())
print(u["Attack"].value_counts())

rows: 2390275 | source IPs: 40
Attack
Benign            2295222
Exploits            31551
Fuzzers             22310
Generic             16560
Reconnaissance      12779
DoS                  5794
Analysis             2299
Backdoor             2169
Shellcode            1427
Worms                 164
Name: count, dtype: int64[pyarrow]
